[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Teoria_General_%28Jupyter.gen%29/Teoria_General_%28%28Jupyter.gen.gen%29/Schrodinger_Paquete_Ondas.ipynb)


# Mecánica Cuántica: Efecto Túnel Dinámico (Paquete de Ondas)
Resolveremos la **Ecuación de Schrödinger Dependiente del Tiempo (TDSE)** en 1D.

$$ i\hbar \frac{\partial \psi}{\partial t} = \left( -\frac{\hbar^2}{2m} \frac{\partial^2}{\partial x^2} + V(x) \right) \psi $$

Usaremos un algoritmo numérico para evolucionar un paquete de ondas Gaussiano libre que viaja hacia la derecha. Cuando colisiona contra una barrera de potencial impenetrable para la física clásica, la onda se divide: una parte es reflejada y otra parte penetra exponencialmente y escapa hacia el otro lado (**Efecto Túnel**).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from IPython.display import HTML

# Parámetros del espacio y tiempo
Nx = 1000       # Nodos espaciales
Lx = 100.0      # Longitud del dominio
dx = Lx / Nx
x = np.linspace(0, Lx, Nx)

dt = 0.05       # Paso temporal
steps = 400     # Número total de frames

# Parámetros físicos (unidades atómicas hbar = 1, m = 1)
x0 = 25.0       # Posición inicial del paquete
sigma = 5.0     # Anchura inicial del paquete
k0 = 2.0        # Momento inicial (velocidad de fase)

# Barrera de potencial V(x)
V = np.zeros(Nx)
V_height = 2.5
bar_start, bar_end = int(Nx*0.45), int(Nx*0.5)
V[bar_start:bar_end] = V_height

# Estado inicial: Paquete Gaussiano normalizado
psi = np.exp(-0.5 * ((x - x0) / sigma)**2) * np.exp(1j * k0 * x)
psi /= np.sqrt(np.sum(np.abs(psi)**2 * dx))

# Método Crank-Nicolson (Unitario e incondicionalmente estable)
# (I + i dt/2 H) psi_new = (I - i dt/2 H) psi_old
alpha = 1j * dt / (4 * dx**2)  # hbar=1, 2m=2 -> m=1

main_diag_A = 1 + 2*alpha + 1j * dt / 2 * V
off_diag_A = -alpha * np.ones(Nx - 1)
A = diags([off_diag_A, main_diag_A, off_diag_A], offsets=[-1, 0, 1], format='csr')

main_diag_B = 1 - 2*alpha - 1j * dt / 2 * V
off_diag_B = alpha * np.ones(Nx - 1)
B = diags([off_diag_B, main_diag_B, off_diag_B], offsets=[-1, 0, 1], format='csr')

# Visualización
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(10, 5))
line, = ax.plot(x, np.abs(psi)**2, color='cyan', lw=2, label='$|\\psi(x)|^2$')
ax.fill_between(x, 0, V / V_height * 0.1, color='red', alpha=0.5, label='Barrera V(x)')
ax.set_ylim(0, 0.1)
ax.set_xlim(0, Lx)
ax.set_title("Efecto Túnel Cuántico: Evolución de la Ecuación de Schrödinger")
ax.legend(loc='upper right')

def update(frame):
    global psi
    # Resolver sistema tridiagonal en cada paso (Crank-Nicolson)
    rhs = B.dot(psi)
    psi = spsolve(A, rhs)
    line.set_ydata(np.abs(psi)**2)
    return line,

# Descomenta las siguientes líneas si ejecutas en un Jupyter real para ver la animación:
# anim = FuncAnimation(fig, update, frames=steps, interval=20, blit=True)
# HTML(anim.to_jshtml())

# Simulamos sin animar para mostrar la foto final aquí:
for _ in range(250):
    rhs = B.dot(psi)
    psi = spsolve(A, rhs)
line.set_ydata(np.abs(psi)**2)
plt.show()
print("Observa cómo la densidad de probabilidad se ha dividido. Una parte ha cruzado mágicamente al lado derecho.")